# Predicting Language Model Memorization from Tokenizer Artifacts

**Setup:** Runtime > Change runtime type > **T4 GPU** > Save

Two configs available:
- `colab_nuke` -- Smoke test (30 synthetic canaries, ~10 min)
- `colab_real` -- Realistic experiment (WikiText-103 corpus, 1000 candidates, ~45 min)

In [ ]:
# Clone repo
!git clone https://github.com/louatiMahmoudAziz/Predicting-Language-Model-Memorization-from-Tokenizer-Artifacts.git
%cd Predicting-Language-Model-Memorization-from-Tokenizer-Artifacts

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q datasets

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## Option A: Smoke test (`colab_nuke`, ~10 min)
Small synthetic corpus, 30 canaries. Use this to verify the pipeline works.

In [ ]:
# Generate nuke data and run
!python scripts/gen_nuke_data.py
!python -m src.run_pipeline --config configs/colab_nuke.yaml

---
## Option B: Realistic experiment (`colab_real`, ~45 min)
WikiText-103 corpus, 1000 candidates (100 canaries + 900 benign strings),
varied repetition schedule [1, 3, 10, 50, 100].

In [ ]:
# Download WikiText-103 + generate canaries and benign candidates
!python scripts/gen_real_data.py

In [ ]:
# Run full pipeline (~45 min on T4)
!python -m src.run_pipeline --config configs/colab_real.yaml

---
## Inspect results

In [ ]:
import pandas as pd
import json

# Change to the run_id you used
RUN_ID = "colab_real"  # or "colab_nuke"

# Pipeline summary
manifest = json.load(open(f"results/{RUN_ID}/{RUN_ID}_pipeline.json"))
print("=== PIPELINE STEPS ===")
for s in manifest["steps"]:
    print(f"  {s['step']:02d}  {s['name']:<22}  {s['status']:<8}  {s.get('elapsed_s', 0):.1f}s")

# Delta BPC distribution
labels = pd.read_parquet(f"labels/{RUN_ID}_labels.parquet")
valid = labels[labels["valid_label"] == True]
print(f"\n=== DELTA BPC (valid rows: {len(valid)} / {len(labels)}) ===")

is_canary = valid["candidate_id"].str.startswith("canary_")
print(f"  Canaries:  mean={valid.loc[is_canary, 'delta_bpc'].mean():.2f}  "
      f"min={valid.loc[is_canary, 'delta_bpc'].min():.2f}  "
      f"max={valid.loc[is_canary, 'delta_bpc'].max():.2f}")
if (~is_canary).any():
    print(f"  Benign:    mean={valid.loc[~is_canary, 'delta_bpc'].mean():.2f}  "
          f"min={valid.loc[~is_canary, 'delta_bpc'].min():.2f}  "
          f"max={valid.loc[~is_canary, 'delta_bpc'].max():.2f}")

In [ ]:
# Evaluation metrics comparison
comp = pd.read_parquet(f"results/{RUN_ID}/eval/comparison.parquet")
display_cols = [c for c in ["task", "model_name", "feature_subset", "n_test", "n_pos",
                            "auroc", "auprc", "reg_pearson_r", "reg_spearman_rho"]
                if c in comp.columns]
comp[display_cols]

---
## Save results to Google Drive (optional)
Uncomment and run to persist results after session ends.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# dst = f"/content/drive/MyDrive/{RUN_ID}_results"
# shutil.copytree(f"results/{RUN_ID}", dst, dirs_exist_ok=True)
# shutil.copytree("labels", f"{dst}/labels", dirs_exist_ok=True)
# shutil.copytree("features", f"{dst}/features", dirs_exist_ok=True)
# print(f"Results saved to Google Drive: {dst}")